In [8]:
# 1. Clone the repository structure without downloading all files (blobs)
!git clone --depth 1 --filter=blob:none --sparse https://github.com/SQL-Storm/SQLStorm.git

# 2. Navigate into the repository
%cd SQLStorm

# 3. Configure Git to only download the specific subdirectory you need
!git sparse-checkout set v1.0/stackoverflow/queries

Cloning into 'SQLStorm'...
remote: Enumerating objects: 54, done.
remote: Counting objects: 100% (54/54), done.
remote: Compressing objects: 100% (50/50), done.
remote: Total 54 (delta 3), reused 46 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (54/54), 12.63 MiB | 37.59 MiB/s, done.
Resolving deltas: 100% (3/3), done.
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 7 (delta 0), reused 7 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (7/7), 5.31 KiB | 5.31 MiB/s, done.
/kaggle/working/SQLStorm
remote: Enumerating objects: 17314, done.
remote: Counting objects: 100% (17314/17314), done.
remote: Compressing objects: 100% (16077/16077), done.
remote: Total 17314 (delta 1235), reused 17308 (delta 1233), pack-reused 0 (from 0)
Receiving objects: 100% (17314/17314), 7.75 MiB | 20.93 MiB/s, done.
Resolving deltas: 100% (1235/1235), done.
Updating files: 100% (18258/18258), don

In [9]:
path="/kaggle/working/SQLStorm/v1.0/stackoverflow/queries"
import pandas as pd
import os


def load_sql_dir(directory):
    """
    Load every .sql file in `directory` into a pandas DataFrame.

    Parameters
    ----------
    directory : str
        Path to the folder containing .sql files.

    Returns
    -------
    pd.DataFrame
        Columns: ['file_name', 'sql_text']
    """
    records = []

    for file_name in sorted(os.listdir(directory)):
        if file_name.lower().endswith(".sql"):
            file_path = os.path.join(directory, file_name)
            try:
                with open(file_path, "r", encoding="utf-8") as f:
                    sql_text = f.read()
                records.append({"file_name": file_name, "sql_text": sql_text})
            except (FileNotFoundError, PermissionError, UnicodeDecodeError) as e:
                print(f"[SKIP] {file_path} → {e}")

    df = pd.DataFrame(records, columns=["file_name", "sql_text"])
    print(f"Loaded {len(df)} .sql files from {directory}")
    return df
df_stackoverflow=load_sql_dir(path)

Loaded 18251 .sql files from /kaggle/working/SQLStorm/v1.0/stackoverflow/queries


In [12]:
# query2vec_inference.py

import re
import sqlparse
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from collections import Counter
from typing import List, Optional

# -------------------------------
# Model Components (exactly as in training)
# -------------------------------

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]


class ATTENTION(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.W1 = nn.Linear(hidden_dim, hidden_dim)
        self.W2 = nn.Linear(hidden_dim, hidden_dim)
        self.V = nn.Linear(hidden_dim, 1)

    def forward(self, hidden, encoder_outputs):
        h = hidden[0]
        forward = h[-2, :, :]
        backward = h[-1, :, :]
        last_hidden = torch.cat([forward, backward], dim=1)
        last_hidden = last_hidden.unsqueeze(1)
        score = self.V(torch.tanh(self.W1(encoder_outputs) + self.W2(last_hidden)))
        attention_weights = torch.softmax(score, dim=1)
        context = torch.sum(attention_weights * encoder_outputs, dim=1)
        return context


class LSTMENCODERDECODER(nn.Module):
    def __init__(self, vocab_size, embed_dim=256, hidden_dim=256, num_layers=1,
                 dropout=0.3, pad_idx=0, use_attention=True, use_vae=False):
        super().__init__()
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.pad_idx = pad_idx
        self.use_attention = use_attention
        self.use_vae = use_vae

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.pos_encoding = PositionalEncoding(embed_dim, max_len=5000)
        self.encoder = nn.LSTM(
            embed_dim, hidden_dim, num_layers=num_layers,
            batch_first=True, bidirectional=True,
            dropout=dropout if num_layers > 1 else 0
        )
        if use_attention:
            self.attention = ATTENTION(hidden_dim * 2)
        if use_vae:
            self.fc_mu = nn.Linear(hidden_dim * 2, hidden_dim * 2)
            self.fc_logvar = nn.Linear(hidden_dim * 2, hidden_dim * 2)
        self.bridge = nn.Linear(hidden_dim * 2, hidden_dim * 2)
        self.decoder = nn.LSTM(
            embed_dim, hidden_dim * 2, num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        self.fc_out = nn.Linear(hidden_dim * 2, vocab_size)
        self.dropout = nn.Dropout(dropout)
        self.layer_norm = nn.LayerNorm(hidden_dim * 2)
        self._init_weights()

    def _init_weights(self):
        for name, param in self.named_parameters():
            if 'weight' in name:
                if param.dim() >= 2:
                    if 'lstm' in name:
                        if 'weight_ih' in name:
                            nn.init.xavier_uniform_(param)
                        elif 'weight_hh' in name:
                            nn.init.orthogonal_(param)
                    else:
                        nn.init.xavier_uniform_(param)
                else:
                    nn.init.ones_(param)
            elif 'bias' in name:
                nn.init.constant_(param, 0.0)

    def forward(self, x, lengths=None, teacher_forcing_ratio=0.5):
        # Not used during inference
        pass

    def encode(self, x, lengths=None):
        embedded = self.dropout(self.embedding(x))
        embedded = self.pos_encoding(embedded)
        if lengths is not None:
            packed = pack_padded_sequence(embedded, lengths.cpu(), batch_first=True, enforce_sorted=False)
            output, hidden = self.encoder(packed)
            output, _ = pad_packed_sequence(output, batch_first=True)
        else:
            output, hidden = self.encoder(embedded)
        return output, hidden

    def get_query_vector(self, x, lengths=None):
        with torch.no_grad():
            encoder_outputs, encoder_hidden = self.encode(x, lengths)
            if self.use_attention:
                query_vec = self.attention(encoder_hidden, encoder_outputs)
            else:
                forward = encoder_hidden[0][-2, :, :]
                backward = encoder_hidden[0][-1, :, :]
                query_vec = torch.cat([forward, backward], dim=1)
            if self.use_vae:
                query_vec = self.fc_mu(query_vec)
            return query_vec

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def init_decoder_hidden(self, context):
        hidden = context.unsqueeze(0).repeat(self.num_layers, 1, 1)
        cell = torch.zeros_like(hidden)
        return (hidden, cell)


# -------------------------------
# Query Preprocessing (identical to notebook)
# -------------------------------

def normalize_sql(sql: str) -> str:
    """Lowercase and collapse all whitespace."""
    if not isinstance(sql, str):
        return ""
    sql = sql.lower().strip()
    sql = re.sub(r'\s+', ' ', sql)
    return sql

# Regex to catch scientific / sneaky numbers (from the notebook)
SNEAKY_NUMBER_PATTERN = re.compile(r'^[-+]?(?:\d+\.\d*|\d*\.\d+|\d+)(?:[eE][-+]?\d+)?$')

def tokenize_sql(sql: str) -> List[str]:
    """
    Tokenise SQL and mask literals.
    Also catches numbers that sqlparse might have missed.
    """
    sql = normalize_sql(sql)
    if not sql:
        return []

    parsed = sqlparse.parse(sql)
    if not parsed:
        return []

    tokens = []
    for token in parsed[0].flatten():
        if token.is_whitespace:
            continue
        if token.ttype in sqlparse.tokens.Number:
            tokens.append('NUM_LITERAL')
        elif token.ttype in sqlparse.tokens.String:
            tokens.append('STR_LITERAL')
        elif token.value.startswith('0x') and len(token.value) > 2:
            tokens.append('HEX_LITERAL')
        else:
            tokens.append(token.value)

    # Second pass: catch sneaky numbers that weren't masked
    fixed_tokens = []
    for tok in tokens:
        if SNEAKY_NUMBER_PATTERN.fullmatch(tok):
            fixed_tokens.append('NUM_LITERAL')
        elif re.fullmatch(r'^[-+]?\d+$', tok):
            fixed_tokens.append('NUM_LITERAL')
        else:
            fixed_tokens.append(tok)

    return fixed_tokens


# -------------------------------
# Inference Class
# -------------------------------

class Query2Vec:
    """
    Loads the trained LSTM auto‑encoder and the exact tokenizer
    used during training, and provides a method to obtain
    embeddings for SQL queries.
    """
    def __init__(
        self,
        model_path: str,
        data_path: str,
        device: Optional[str] = None
    ):
        """
        Args:
            model_path: Path to the trained model checkpoint
                        (e.g. "best_model.pth").
            data_path:  Path to the final_encoded_data.pt file
                        that contains token2id.
            device:     'cpu', 'cuda', or None (auto‑detect).
        """
        # Load tokenizer and vocabulary
        data = torch.load(data_path, map_location='cpu')
        self.token2id = data['token2id']
        self.id2token = data['id2token']
        self.vocab_size = len(self.token2id)
        self.pad_idx = self.token2id.get('<PAD>', 0)
        self.unk_idx = self.token2id.get('<UNK>', 2)

        # Hard‑coded model architecture (must match training)
        embed_dim = 64
        hidden_dim = 64
        num_layers = 2
        dropout = 0.3
        use_attention = True
        use_vae = False

        # Build model
        self.model = LSTMENCODERDECODER(
            vocab_size=self.vocab_size,
            embed_dim=embed_dim,
            hidden_dim=hidden_dim,
            num_layers=num_layers,
            dropout=dropout,
            pad_idx=self.pad_idx,
            use_attention=use_attention,
            use_vae=use_vae
        )

        # Load weights
        checkpoint = torch.load(model_path, map_location='cpu')
        # The notebook saved 'model_state_dict', not the whole model
        if 'model_state_dict' in checkpoint:
            self.model.load_state_dict(checkpoint['model_state_dict'])
        else:
            self.model.load_state_dict(checkpoint)

        # Set device
        if device is None:
            self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        else:
            self.device = torch.device(device)
        self.model.to(self.device)
        self.model.eval()

        # Embedding dimension
        self.embedding_dim = hidden_dim * 2   # bidirectional → 128

    def _preprocess_sql(self, sql: str) -> List[int]:
        """Tokenise and convert to integer IDs."""
        if not isinstance(sql, str) or not sql.strip():
            return []
        tokens = tokenize_sql(sql)
        ids = [self.token2id.get(tok, self.unk_idx) for tok in tokens]
        return ids

    def get_embeddings(
        self,
        df: pd.DataFrame,
        query_column: str,
        batch_size: int = 64
    ) -> np.ndarray:
        """
        Compute query embeddings for a DataFrame.

        Args:
            df:           Pandas DataFrame containing SQL queries.
            query_column: Name of the column with SQL strings.
            batch_size:   Batch size for inference.

        Returns:
            NumPy array of shape (len(df), embedding_dim).
            Rows where the query is empty/invalid are filled with zeros.
        """
        all_ids = []
        empty_mask = []
        for sql in df[query_column]:
            ids = self._preprocess_sql(sql)
            if ids:
                all_ids.append(torch.tensor(ids, dtype=torch.long))
                empty_mask.append(False)
            else:
                # Placeholder for empty queries
                all_ids.append(torch.tensor([self.pad_idx], dtype=torch.long))
                empty_mask.append(True)

        # Determine lengths before padding
        lengths = torch.tensor([len(seq) for seq in all_ids], dtype=torch.long)

        # Pad sequences
        padded = torch.nn.utils.rnn.pad_sequence(
            all_ids, batch_first=True, padding_value=self.pad_idx
        )

        # Collect embeddings
        embeddings_list = []
        with torch.no_grad():
            for i in range(0, len(padded), batch_size):
                batch_x = padded[i:i+batch_size].to(self.device)
                batch_len = lengths[i:i+batch_size].to(self.device)
                vecs = self.model.get_query_vector(batch_x, batch_len)
                embeddings_list.append(vecs.cpu().numpy())

        embeddings = np.concatenate(embeddings_list, axis=0)

        # Set empty queries to zero vector
        empty_idx = np.where(empty_mask)[0]
        if len(empty_idx) > 0:
            embeddings[empty_idx] = np.zeros(self.embedding_dim, dtype=np.float32)

        return embeddings.astype(np.float32)


MODEL_PATH = "/kaggle/input/datasets/balasiva2007/trained-model/final_model.pth"  
DATA_PATH = "/kaggle/input/datasets/balasiva2007/encoded-data/final_encoded_data.pt"

# Create inference object
q2v = Query2Vec(MODEL_PATH, DATA_PATH)



# Get embeddings
embeddings = q2v.get_embeddings(df_stackoverflow, query_column="sql_text")
print("Embeddings shape:", embeddings.shape)  # (3, 128)
print("First embedding:\n", embeddings[0])

RuntimeError: Error(s) in loading state_dict for LSTMENCODERDECODER:
	size mismatch for embedding.weight: copying a param with shape torch.Size([7184, 128]) from checkpoint, the shape in current model is torch.Size([7184, 64]).
	size mismatch for pos_encoding.pe: copying a param with shape torch.Size([1, 5000, 128]) from checkpoint, the shape in current model is torch.Size([1, 5000, 64]).
	size mismatch for encoder.weight_ih_l0: copying a param with shape torch.Size([512, 128]) from checkpoint, the shape in current model is torch.Size([256, 64]).
	size mismatch for encoder.weight_hh_l0: copying a param with shape torch.Size([512, 128]) from checkpoint, the shape in current model is torch.Size([256, 64]).
	size mismatch for encoder.bias_ih_l0: copying a param with shape torch.Size([512]) from checkpoint, the shape in current model is torch.Size([256]).
	size mismatch for encoder.bias_hh_l0: copying a param with shape torch.Size([512]) from checkpoint, the shape in current model is torch

In [15]:
# query2vec_inference.py

import re
import sqlparse
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from collections import Counter
from typing import List, Optional

# -------------------------------
# Model Components (identical to training)
# -------------------------------

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]


class ATTENTION(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.W1 = nn.Linear(hidden_dim, hidden_dim)
        self.W2 = nn.Linear(hidden_dim, hidden_dim)
        self.V = nn.Linear(hidden_dim, 1)

    def forward(self, hidden, encoder_outputs):
        h = hidden[0]
        forward = h[-2, :, :]
        backward = h[-1, :, :]
        last_hidden = torch.cat([forward, backward], dim=1)
        last_hidden = last_hidden.unsqueeze(1)
        score = self.V(torch.tanh(self.W1(encoder_outputs) + self.W2(last_hidden)))
        attention_weights = torch.softmax(score, dim=1)
        context = torch.sum(attention_weights * encoder_outputs, dim=1)
        return context


class LSTMENCODERDECODER(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers,
                 dropout=0.3, pad_idx=0, use_attention=True, use_vae=False):
        super().__init__()
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.pad_idx = pad_idx
        self.use_attention = use_attention
        self.use_vae = use_vae

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.pos_encoding = PositionalEncoding(embed_dim, max_len=5000)
        self.encoder = nn.LSTM(
            embed_dim, hidden_dim, num_layers=num_layers,
            batch_first=True, bidirectional=True,
            dropout=dropout if num_layers > 1 else 0
        )
        if use_attention:
            self.attention = ATTENTION(hidden_dim * 2)
        if use_vae:
            self.fc_mu = nn.Linear(hidden_dim * 2, hidden_dim * 2)
            self.fc_logvar = nn.Linear(hidden_dim * 2, hidden_dim * 2)
        self.bridge = nn.Linear(hidden_dim * 2, hidden_dim * 2)
        self.decoder = nn.LSTM(
            embed_dim, hidden_dim * 2, num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        self.fc_out = nn.Linear(hidden_dim * 2, vocab_size)
        self.dropout = nn.Dropout(dropout)
        self.layer_norm = nn.LayerNorm(hidden_dim * 2)
        self._init_weights()

    def _init_weights(self):
        for name, param in self.named_parameters():
            if 'weight' in name:
                if param.dim() >= 2:
                    if 'lstm' in name:
                        if 'weight_ih' in name:
                            nn.init.xavier_uniform_(param)
                        elif 'weight_hh' in name:
                            nn.init.orthogonal_(param)
                    else:
                        nn.init.xavier_uniform_(param)
                else:
                    nn.init.ones_(param)
            elif 'bias' in name:
                nn.init.constant_(param, 0.0)

    def forward(self, x, lengths=None, teacher_forcing_ratio=0.5):
        pass  # not used during inference

    def encode(self, x, lengths=None):
        embedded = self.dropout(self.embedding(x))
        embedded = self.pos_encoding(embedded)
        if lengths is not None:
            packed = pack_padded_sequence(embedded, lengths.cpu(), batch_first=True, enforce_sorted=False)
            output, hidden = self.encoder(packed)
            output, _ = pad_packed_sequence(output, batch_first=True)
        else:
            output, hidden = self.encoder(embedded)
        return output, hidden

    def get_query_vector(self, x, lengths=None):
        with torch.no_grad():
            encoder_outputs, encoder_hidden = self.encode(x, lengths)
            if self.use_attention:
                query_vec = self.attention(encoder_hidden, encoder_outputs)
            else:
                forward = encoder_hidden[0][-2, :, :]
                backward = encoder_hidden[0][-1, :, :]
                query_vec = torch.cat([forward, backward], dim=1)
            if self.use_vae:
                query_vec = self.fc_mu(query_vec)
            return query_vec

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def init_decoder_hidden(self, context):
        hidden = context.unsqueeze(0).repeat(self.num_layers, 1, 1)
        cell = torch.zeros_like(hidden)
        return (hidden, cell)


# -------------------------------
# Query Preprocessing (identical to notebook)
# -------------------------------

def normalize_sql(sql: str) -> str:
    if not isinstance(sql, str):
        return ""
    sql = sql.lower().strip()
    sql = re.sub(r'\s+', ' ', sql)
    return sql

SNEAKY_NUMBER_PATTERN = re.compile(r'^[-+]?(?:\d+\.\d*|\d*\.\d+|\d+)(?:[eE][-+]?\d+)?$')

def tokenize_sql(sql: str) -> List[str]:
    sql = normalize_sql(sql)
    if not sql:
        return []
    parsed = sqlparse.parse(sql)
    if not parsed:
        return []
    tokens = []
    for token in parsed[0].flatten():
        if token.is_whitespace:
            continue
        if token.ttype in sqlparse.tokens.Number:
            tokens.append('NUM_LITERAL')
        elif token.ttype in sqlparse.tokens.String:
            tokens.append('STR_LITERAL')
        elif token.value.startswith('0x') and len(token.value) > 2:
            tokens.append('HEX_LITERAL')
        else:
            tokens.append(token.value)
    fixed_tokens = []
    for tok in tokens:
        if SNEAKY_NUMBER_PATTERN.fullmatch(tok):
            fixed_tokens.append('NUM_LITERAL')
        elif re.fullmatch(r'^[-+]?\d+$', tok):
            fixed_tokens.append('NUM_LITERAL')
        else:
            fixed_tokens.append(tok)
    return fixed_tokens


# -------------------------------
# Inference Class (auto‑detects model dimensions)
# -------------------------------

class Query2Vec:
    def __init__(self, model_path: str, data_path: str, device: Optional[str] = None):
        # 1. Load tokenizer
        data = torch.load(data_path, map_location='cpu')
        self.token2id = data['token2id']
        self.id2token = data['id2token']
        self.vocab_size = len(self.token2id)
        self.pad_idx = self.token2id.get('<PAD>', 0)
        self.unk_idx = self.token2id.get('<UNK>', 2)

        # 2. Load checkpoint
        checkpoint = torch.load(model_path, map_location='cpu')
        state_dict = checkpoint.get('model_state_dict', checkpoint)

        # 3. Auto‑detect architecture from the state dict
        embed_dim = state_dict['embedding.weight'].shape[1]
        # hidden_dim: encoder.weight_ih_l0 is (4*hidden_dim, embed_dim)
        hidden_dim = state_dict['encoder.weight_ih_l0'].shape[0] // 4

        # Count LSTM layers: look for 'weight_ih_l{idx}' in encoder (no reverse)
        num_layers = 0
        while f'encoder.weight_ih_l{num_layers}' in state_dict:
            num_layers += 1

        # Detect dropout (optional; we'll just use 0.3 as default)
        dropout = 0.3

        # Detect attention / VAE by key presence
        use_attention = any(k.startswith('attention.') for k in state_dict)
        use_vae = any(k.startswith('fc_mu.') for k in state_dict)

        # 4. Build model with detected parameters
        self.model = LSTMENCODERDECODER(
            vocab_size=self.vocab_size,
            embed_dim=embed_dim,
            hidden_dim=hidden_dim,
            num_layers=num_layers,
            dropout=dropout,
            pad_idx=self.pad_idx,
            use_attention=use_attention,
            use_vae=use_vae
        )
        self.model.load_state_dict(state_dict)

        # 5. Device and eval mode
        if device is None:
            self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        else:
            self.device = torch.device(device)
        self.model.to(self.device)
        self.model.eval()

        # Embedding dimension = hidden_dim * 2 (bidirectional)
        self.embedding_dim = hidden_dim * 2

    def _preprocess_sql(self, sql: str) -> List[int]:
        if not isinstance(sql, str) or not sql.strip():
            return []
        tokens = tokenize_sql(sql)
        ids = [self.token2id.get(tok, self.unk_idx) for tok in tokens]
        return ids

    def get_embeddings(self, df: pd.DataFrame, query_column: str, batch_size: int = 64) -> np.ndarray:
        all_ids = []
        empty_mask = []
        for sql in df[query_column]:
            ids = self._preprocess_sql(sql)
            if ids:
                all_ids.append(torch.tensor(ids, dtype=torch.long))
                empty_mask.append(False)
            else:
                all_ids.append(torch.tensor([self.pad_idx], dtype=torch.long))
                empty_mask.append(True)

        lengths = torch.tensor([len(seq) for seq in all_ids], dtype=torch.long)
        padded = torch.nn.utils.rnn.pad_sequence(all_ids, batch_first=True, padding_value=self.pad_idx)

        embeddings_list = []
        with torch.no_grad():
            for i in range(0, len(padded), batch_size):
                batch_x = padded[i:i+batch_size].to(self.device)
                batch_len = lengths[i:i+batch_size].to(self.device)
                vecs = self.model.get_query_vector(batch_x, batch_len)
                embeddings_list.append(vecs.cpu().numpy())

        embeddings = np.concatenate(embeddings_list, axis=0)
        empty_idx = np.where(empty_mask)[0]
        if len(empty_idx) > 0:
            embeddings[empty_idx] = np.zeros(self.embedding_dim, dtype=np.float32)
        return embeddings.astype(np.float32)


q2v = Query2Vec(MODEL_PATH, DATA_PATH)


embeddings = q2v.get_embeddings(df_stackoverflow, query_column="sql_text")
print("Embeddings shape:", embeddings.shape)
print("First embedding:\n", embeddings[0])

Embeddings shape: (18251, 256)
First embedding:
 [-5.10667562e-01 -5.15960678e-02 -1.74352646e-01  1.61755264e-01
 -1.75611213e-01 -6.86156452e-02  6.93842590e-01  5.88345826e-01
  3.34079146e-01  3.86387184e-02 -1.94097877e-01  7.44364798e-01
 -4.01859045e-01  6.16030753e-01  7.44413912e-01  6.56459928e-01
  1.74544066e-01 -8.16097409e-02 -3.53374258e-02  3.20208371e-01
  7.44807482e-01 -2.27905706e-01  4.90628444e-02 -3.21787655e-01
  2.93190718e-01 -1.00735258e-02  2.90542543e-02  5.75055629e-02
 -5.79384208e-01  2.63019979e-01 -9.09774899e-02  7.35873803e-02
 -2.91562378e-01  1.20115140e-02 -1.19304880e-01  4.77470644e-03
  2.58912027e-01 -2.46909261e-02  6.09108984e-01  8.69576540e-03
  1.24436751e-01 -2.27979556e-01  3.06703662e-03  1.04775883e-01
 -2.79562455e-03 -3.14033747e-01  1.90102220e-01 -2.97923863e-01
  2.02256396e-01  4.31262851e-02 -4.39067632e-02  3.38624232e-02
 -4.44610089e-01 -9.48235393e-01 -6.45049289e-03  4.15814035e-02
 -8.19572061e-02  5.03607243e-02 -3.98958

In [16]:
np.save("lstm_embeds.npy",embeddings)

In [17]:
!ls

lstm_embeds.npy  v1.0
